## Standalone CPT Processing Pipeline (TXT + PDF)

This notebook provides a **general, reusable pipeline** for converting existing **TXT** and **PDF** files into **CPT JSONL**.

### Purpose

- Produce a clean, consistent CPT dataset (one JSON object per line)
- Remove common noise/boilerplate (especially from PDFs)
- Optionally split long text into chunks

### What you configure

- **Source labels**: `TXT_SOURCE_NAME`, `PDF_SOURCE_NAME`
- **TXT cleaning ruleset**: `TXT_RULESET` (you can add rulesets in `TXT_RULESETS`)
- **Paths**: input folders + output JSONL paths

### Processing flow

1) remove noise
2) clean / normalize text
3) optional chunking
4) format into CPT records
5) write JSONL

### Notes

- **PDFs**: extraction is page-based; default output is **one record per page**, with header/footer noise removed via frequency detection.
- **TXTs**: cleaning is ruleset-driven so you can reuse the same pipeline for multiple sources without changing the output schema.

## Cell 1 — Imports

TXT processing uses only standard library.

PDF processing needs:
- `pymupdf` (PyMuPDF) for text extraction
- optional: `pdfplumber` for table extraction


In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

try:
    import fitz  # PyMuPDF
    HAS_PYMUPDF = True
except Exception:
    HAS_PYMUPDF = False

try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except Exception:
    HAS_PDFPLUMBER = False

print("HAS_PYMUPDF:", HAS_PYMUPDF)
print("HAS_PDFPLUMBER:", HAS_PDFPLUMBER)

## Cell 2 — CPT record schema

CPT rules:
- always set `text`, `input`, `content` = the same string
- always include `source` and `doc_id`
- optional metadata allowed: `url`, `title`, `filename`, `page`


In [ ]:
def make_cpt_record(
    text: str,
    source: str,
    doc_id: str,
    *,
    url: str | None = None,
    title: str | None = None,
    filename: str | None = None,
    page: int | None = None,
) -> dict:
    """Build one CPT chunk record."""
    rec = {
        "text": text,
        "input": text,
        "content": text,
        "source": source,
        "doc_id": doc_id,
    }
    if url is not None:
        rec["url"] = url
    if title is not None:
        rec["title"] = title
    if filename is not None:
        rec["filename"] = filename
    if page is not None:
        rec["page"] = page
    return rec

print("make_cpt_record ready")

## Cell 3 — Chunking

Important:
- chunk size is **character-based**, not token-based
- respects blank-line paragraphs


In [ ]:
def chunk_text(text: str, max_chars: int) -> list[str]:
    """
    Split text into chunks of at most max_chars, respecting blank-line paragraphs.
    If max_chars <= 0 or text is shorter, returns [text] (or [] if empty).
    """
    if not text:
        return []
    text = text.strip()
    if not max_chars or max_chars <= 0 or len(text) <= max_chars:
        return [text]

    out: list[str] = []
    paragraphs = text.split("\n\n")
    current = ""
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if len(current) + len(p) + 2 <= max_chars:
            current = (current + "\n\n" + p).strip() if current else p
        else:
            if current:
                out.append(current)
            if len(p) > max_chars:
                for i in range(0, len(p), max_chars):
                    chunk = p[i : i + max_chars].strip()
                    if chunk:
                        out.append(chunk)
                current = ""
            else:
                current = p
    if current:
        out.append(current)
    return out

print("chunk_text ready")

## Cell 4 — PDF extract + noise detection + cleaning

Noise removal logic:
- remove standalone numbers (page numbers)
- detect repeating lines across pages (header/footer) and remove them
- normalize newlines


In [ ]:
NOISE_LINE_PATTERN = re.compile(r"^\s*\d{1,4}\s*$", re.MULTILINE)

def table_to_text(table: list[list]) -> str:
    """Convert table (list of rows) to readable text (pipe-separated)."""
    if not table:
        return ""
    return "\n".join(" | ".join(str(c or "").strip() for c in row) for row in table)

def extract_tables_from_page(pdf_path: str, page_num: int) -> list[str]:
    """Extract tables from one page using pdfplumber."""
    if not HAS_PDFPLUMBER:
        return []
    out = []
    try:
        with pdfplumber.open(pdf_path) as doc:
            if 1 <= page_num <= len(doc.pages):
                for t in doc.pages[page_num - 1].extract_tables() or []:
                    txt = table_to_text(t)
                    if txt.strip():
                        out.append(txt)
    except Exception:
        pass
    return out

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """Extract text per page; append [Table] blocks if pdfplumber available."""
    if not HAS_PYMUPDF:
        return []
    out = []
    try:
        doc = fitz.open(pdf_path)
        for i in range(len(doc)):
            text = doc[i].get_text()
            if HAS_PDFPLUMBER:
                tables = extract_tables_from_page(pdf_path, i + 1)
                if tables:
                    text = (text.strip() if text else "") + "\n\n[Table]\n" + "\n\n[Table]\n".join(tables)
            if text and text.strip():
                out.append({"page": i + 1, "text": text.strip()})
        doc.close()
    except Exception:
        return []
    return out

def detect_noise_lines(
    pages: list[dict],
    min_freq_ratio: float = 0.3,
    max_line_len: int = 120,
) -> set[str]:
    """Detect running headers/footers by frequency across pages."""
    if not pages:
        return set()
    total = len(pages)
    cnt: Counter = Counter()
    for item in pages:
        seen = set()
        for line in item["text"].split("\n"):
            s = line.strip()
            if s and s not in seen:
                cnt[s] += 1
                seen.add(s)
    threshold = max(2, int(total * min_freq_ratio))
    return {line for line, c in cnt.items() if c >= threshold and len(line) <= max_line_len}

def clean_pdf_text(text: str, noise_lines: set[str] | None = None) -> str:
    """Normalize whitespace, remove standalone page numbers and detected noise lines."""
    if not text or not text.strip():
        return ""
    text = re.sub(r"\r\n", "\n", text)
    text = NOISE_LINE_PATTERN.sub("", text)
    if noise_lines:
        for line in noise_lines:
            text = re.sub(r"(?m)^" + re.escape(line) + r"\s*$", "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

print("PDF utils ready")

## Cell 5 — General TXT → CPT (ruleset-based)

This is **one TXT processing function** that can handle multiple sources by selecting a **ruleset**.

Flow:

1) parse optional `URL:` / `Title:` header lines
2) compute `doc_id` from cleaned title (or filename fallback)
3) clean body using ruleset:
   - optional title suffix removal
   - optional footer cutoff marker
   - optional leading-noise removal (`SHARE`, date-only line, duplicated title)
4) optional chunking + minimum chunk filter
5) output CPT records (`text/input/content/source/doc_id` + optional `filename`)

In [ ]:
# TXT rulesets: add more as needed.
# "default_news" is a practical preset for common news/article TXT noise.

DATE_LINE = re.compile(
    r"^\s*(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s*\d{4}\s*$",
    re.IGNORECASE,
)

TXT_RULESETS = {
    # Default: remove a known title suffix + footer marker + leading SHARE/date/duplicate-title.
    "default_news": {
        "title_suffix_regex": r"\s*[_\|]\s*International Rice Research Institute\s*$",
        "footer_start_regex": r"\n\s*For more information, please contact\s*:?\s*\n",
        "drop_leading_share": True,
        "drop_leading_date_line": True,
        "drop_leading_duplicate_title": True,
    },
    # Minimal ruleset: no suffix removal, no footer marker; only whitespace normalization.
    "generic": {
        "title_suffix_regex": "",
        "footer_start_regex": "",
        "drop_leading_share": False,
        "drop_leading_date_line": False,
        "drop_leading_duplicate_title": False,
    },
}


def parse_txt_header(raw: str) -> tuple[str, str, str]:
    """Parse optional URL/Title header lines. Returns (url, title, body)."""
    raw = (raw or "").strip()
    url = ""
    title = ""
    body = raw
    if raw.startswith("URL:"):
        lines = raw.split("\n")
        if len(lines) >= 1:
            url = lines[0].replace("URL:", "").strip()
        if len(lines) >= 2 and lines[1].strip().lower().startswith("title:"):
            title = lines[1].replace("Title:", "").strip()
        if len(lines) > 2:
            body = "\n".join(lines[2:]).strip()
    return url, title, body


def clean_title_for_id(title: str, *, ruleset: dict) -> str:
    """Remove configured suffix, normalize whitespace."""
    if not title:
        return "unknown"
    t = title.strip()
    suffix = (ruleset.get("title_suffix_regex") or "").strip()
    if suffix:
        t = re.sub(suffix, "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"\s+", " ", t)
    return t or "unknown"


def safe_doc_id(clean_title: str, *, max_len: int = 100) -> str:
    """Slug from cleaned title (same style as our doc_id convention)."""
    s = (clean_title or "").lower().strip()
    s = re.sub(r"[^\w\s\-]", " ", s)
    s = re.sub(r"\s+", "_", s).strip("_")
    return (s[:max_len] if max_len else s) or "unknown"


def clean_txt_body(body: str, *, clean_title: str, ruleset: dict) -> str:
    """Apply footer cutoff + leading-noise stripping + whitespace normalization."""
    if not body:
        return ""

    footer_pat = (ruleset.get("footer_start_regex") or "").strip()
    if footer_pat:
        footer_re = re.compile(footer_pat, re.IGNORECASE)
        if footer_re.search(body):
            body = footer_re.split(body)[0].strip()

    lines = body.split("\n")
    out = []
    skip_leading = True
    for ln in lines:
        s = ln.strip()
        if skip_leading:
            if not s:
                continue
            if ruleset.get("drop_leading_share") and s.upper() == "SHARE":
                continue
            if ruleset.get("drop_leading_date_line") and DATE_LINE.match(s):
                continue
            if ruleset.get("drop_leading_duplicate_title") and clean_title and s == clean_title:
                continue
            skip_leading = False
        out.append(ln)

    body = "\n".join(out).strip()

    # Normalize: strip each line; collapse multiple blank lines to a single blank line
    lines = [ln.strip() for ln in body.split("\n")]
    out = []
    prev_blank = False
    for ln in lines:
        is_blank = not ln
        if is_blank and prev_blank:
            continue
        prev_blank = is_blank
        out.append(ln)

    return "\n".join(out).strip()


def txt_to_cpt_records(
    txt_path: str,
    *,
    source_name: str,
    ruleset_name: str = "generic",
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
) -> list[dict]:
    """General TXT -> CPT: parse -> clean -> optional chunk -> CPT records."""
    ruleset = TXT_RULESETS.get(ruleset_name) or TXT_RULESETS["generic"]

    path = Path(txt_path)
    if not path.is_file() or path.suffix.lower() != ".txt":
        return []

    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        return []

    url, title, body = parse_txt_header(raw)
    clean_title = clean_title_for_id(title, ruleset=ruleset)
    cleaned = clean_txt_body(body, clean_title=clean_title, ruleset=ruleset)

    if not cleaned or len(cleaned) < min_chunk_chars:
        return []

    base_id = safe_doc_id(clean_title, max_len=100) if clean_title else re.sub(r"[^\w\-.]", "_", path.stem)[:100]
    extra = {"filename": path.name}

    if max_chunk_chars > 0 and len(cleaned) > max_chunk_chars:
        chunks = chunk_text(cleaned, max_chunk_chars)
        out: list[dict] = []
        for j, ch in enumerate(chunks):
            if len(ch) < min_chunk_chars:
                continue
            out.append(make_cpt_record(ch, source_name, f"{base_id}_chunk_{j}", **extra))
        return out

    return [make_cpt_record(cleaned, source_name, base_id, **extra)]


print("General TXT utils ready. Available TXT_RULESETS:", sorted(TXT_RULESETS.keys()))


## Cell 6 — General PDF → CPT (copied logic)

This is the general PDF processor (same logic as the project’s PDF pipeline), and it works for **any** PDF.

Default behavior matches our project:

- **one CPT record per PDF page** (`doc_id = {pdf_stem}_page_{page}`)
- optional additional chunking within a page if `max_chunk_chars > 0` (`..._chunk_{j}`)

Noise removal is handled by the PDF utilities in Cell 4:

- extract page text
- detect repeating header/footer lines
- clean each page by removing detected boilerplate + standalone page-number lines

In [ ]:
def pdf_to_cpt_records(
    pdf_path: str,
    *,
    source_name: str,
    min_chunk_chars: int = 100,
    max_chunk_chars: int = 0,
    chunk_by_page: bool = True,
) -> list[dict]:
    """General PDF -> CPT: extract -> detect noise -> clean per page -> CPT records."""
    pages = extract_text_from_pdf(pdf_path)
    if not pages:
        return []
    noise_lines = detect_noise_lines(pages)

    records: list[dict] = []
    base_id = Path(pdf_path).stem
    fname = Path(pdf_path).name

    for item in pages:
        page_num = item["page"]
        text = clean_pdf_text(item["text"], noise_lines)
        if len(text) < min_chunk_chars:
            continue

        if chunk_by_page and not max_chunk_chars:
            records.append(
                make_cpt_record(
                    text,
                    source_name,
                    f"{base_id}_page_{page_num}",
                    filename=fname,
                    page=page_num,
                )
            )
        else:
            chunks = chunk_text(text, max_chunk_chars) if max_chunk_chars else [text]
            for j, ch in enumerate(chunks):
                if len(ch) < min_chunk_chars:
                    continue
                records.append(
                    make_cpt_record(
                        ch,
                        source_name,
                        f"{base_id}_page_{page_num}_chunk_{j}",
                        filename=fname,
                        page=page_num,
                    )
                )

    return records


print("General PDF utils ready")

## Cell 7 — Write JSONL

Same behavior as our services: one JSON object per line, UTF-8.


In [ ]:
def write_jsonl(records: list[dict], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("write_jsonl ready")

## Cell 8 — End-to-end runner (general)

This runner is general:

- TXT folder → CPT JSONL (choose `TXT_RULESET` + `TXT_SOURCE_NAME`)
- PDF folder → CPT JSONL (choose `PDF_SOURCE_NAME`)

No scraping needed.

Adjust only:

- input folders
- `TXT_RULESET` / `TXT_SOURCE_NAME`
- `PDF_SOURCE_NAME`
- `MIN_CHUNK_CHARS` / `MAX_CHUNK_CHARS`
- output paths

In [ ]:
TXT_DIR = Path("data/txt")
PDF_DIR = Path("data/pdfs")

# Choose source names for metadata (free text label)
TXT_SOURCE_NAME = "source_txt"     # e.g. "source_txt"
PDF_SOURCE_NAME = "source_pdf"     # e.g. "source_pdf"

# Choose TXT cleaning ruleset
TXT_RULESET = "default_news"       # available: "default_news", "generic" (add more in TXT_RULESETS)

OUT_DIR = Path("notebooks/output")
OUT_TXT_JSONL = OUT_DIR / "txt_corpus.jsonl"
OUT_PDF_JSONL = OUT_DIR / "pdf_corpus.jsonl"

MIN_CHUNK_CHARS = 100
MAX_CHUNK_CHARS = 0  # set >0 to enable chunking

print("TXT_DIR exists:", TXT_DIR.is_dir(), TXT_DIR)
print("PDF_DIR exists:", PDF_DIR.is_dir(), PDF_DIR)
print("TXT_RULESET:", TXT_RULESET)
print("TXT_SOURCE_NAME:", TXT_SOURCE_NAME)
print("PDF_SOURCE_NAME:", PDF_SOURCE_NAME)

# TXT folder -> CPT
if TXT_DIR.is_dir():
    txt_files = sorted(TXT_DIR.glob("*.txt"))
    txt_records: list[dict] = []
    for p in txt_files:
        txt_records.extend(
            txt_to_cpt_records(
                str(p),
                source_name=TXT_SOURCE_NAME,
                ruleset_name=TXT_RULESET,
                min_chunk_chars=MIN_CHUNK_CHARS,
                max_chunk_chars=MAX_CHUNK_CHARS,
            )
        )
    write_jsonl(txt_records, OUT_TXT_JSONL)
    print("TXT records:", len(txt_records), "->", OUT_TXT_JSONL)
else:
    print("Skip TXT: folder not found")

# PDF folder -> CPT (per page by default)
if PDF_DIR.is_dir():
    if not HAS_PYMUPDF:
        print("Skip PDFs: install PyMuPDF -> pip install pymupdf")
    else:
        pdf_files = sorted(PDF_DIR.glob("*.pdf"))
        pdf_records: list[dict] = []
        for p in pdf_files:
            pdf_records.extend(
                pdf_to_cpt_records(
                    str(p),
                    source_name=PDF_SOURCE_NAME,
                    min_chunk_chars=MIN_CHUNK_CHARS,
                    max_chunk_chars=MAX_CHUNK_CHARS,
                    chunk_by_page=True,
                )
            )
        write_jsonl(pdf_records, OUT_PDF_JSONL)
        print("PDF records:", len(pdf_records), "->", OUT_PDF_JSONL)
else:
    print("Skip PDFs: folder not found")
